## 1. Install dependencies

In [ ]:
REPO_URL = "https://github.com/eddyliu5/aspire.git"

import os
if not os.path.isdir("aspire"):
    !git clone -b finetuning {REPO_URL}
%cd aspire
!pip install -e . -q
!pip install xgboost -q

## 2. Imports & config

In [ ]:
import os, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

from aspire import AspireModel  
CKPT_FILE   = "best_model.pt"
DEVICE      = "cuda"                # change to "cpu" if no GPU
SEED        = 42
MAX_ROWS    = 10_000
NUM_EPOCHS  = 100
BATCH_SIZE  = 32
NUM_SUPPORT = 15

# ── Pick a CARTE dataset ──────────────────────────────────────────
# Options: "spotify", "us_accidents_severity", "nba_draft",
#          "ramen_ratings", "chocolate_bar_ratings", "roger_ebert",
#          "michelin", "coffee_ratings", "whisky", "yelp", "zomato"
DATASET = "ramen_ratings"

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. CARTE dataset registry

Each entry defines the target column, a dataset description, optional label remapping, and per-feature descriptions used by the ASPIRE text encoder.

In [ ]:
CARTE_DATASETS = {
    "ramen_ratings": {
        "target_col": "Stars",
        "label_map": {"0": "low_rated", "1": "high_rated"},
        "description": (
            "Consumer ratings of instant ramen noodle products from around the world. "
            "The task is to predict whether a ramen product receives a high star rating."
        ),
        "feature_descriptions": {
            "Brand":   "The brand or manufacturer of the instant ramen product.",
            "Variety": "The specific product name or flavor.",
            "Style":   "The packaging format: Pack, Bowl, Cup, Tray, or Box.",
            "Country": "The country where the ramen product was manufactured.",
            "Stars":   "Whether the ramen product received a high or low rating.",
        },
    },
    "chocolate_bar_ratings": {
        "target_col": "Rating",
        "label_map": {"0": "low_quality", "1": "high_quality"},
        "description": (
            "Expert ratings of single-origin craft chocolate bars. "
            "The task is to predict whether a chocolate bar receives a high quality rating."
        ),
        "feature_descriptions": {
            "Company_(Manufacturer)":           "The company that produced the chocolate bar.",
            "Company_Location":                 "The country where the company is headquartered.",
            "Review_Date":                      "The year in which the bar was reviewed.",
            "Country_of_Bean_Origin":           "The country where the cacao beans were grown.",
            "Specific_Bean_Origin_or_Bar_Name": "The specific growing region or bar name.",
            "Cocoa_Percent":                    "The percentage of cocoa content by weight.",
            "Ingredients":                      "The ingredients used in the chocolate bar.",
            "Most_Memorable_Characteristics":   "The most memorable flavor notes.",
            "Rating":                           "Whether the bar was rated high or low quality.",
        },
    },
    "roger_ebert": {
        "target_col": "critic_rating",
        "label_map": {"0": "negative_review", "1": "positive_review"},
        "description": (
            "Movie reviews by film critic Roger Ebert. "
            "The task is to predict whether Ebert gave a movie a positive or negative review."
        ),
        "feature_descriptions": {
            "movie_name":    "The title of the movie.",
            "year":          "The year in which the movie was released.",
            "directors":     "The director or directors of the film.",
            "actors":        "The lead actors in the movie.",
            "genre":         "The genre or combination of genres.",
            "pg_rating":     "The Motion Picture Association content rating.",
            "duration":      "The runtime of the movie in minutes.",
            "critic_rating": "Whether Roger Ebert gave a positive or negative review.",
        },
    },
    "michelin": {
        "target_col": "Award",
        "label_map": {"0": "no_star", "1": "michelin_starred"},
        "description": (
            "Restaurant listings evaluated by the Michelin Guide. "
            "The task is to predict whether a restaurant has received a Michelin star."
        ),
        "feature_descriptions": {
            "Name":        "The name of the restaurant.",
            "Address":     "The full street address of the restaurant.",
            "Location":    "The city or region where the restaurant is located.",
            "Cuisine":     "The style or type of cuisine served.",
            "Longitude":   "Geographic longitude coordinate.",
            "Latitude":    "Geographic latitude coordinate.",
            "Website_Url": "The official website URL.",
            "Award":       "Whether the restaurant has a Michelin star or not.",
        },
    },
    "nba_draft": {
        "target_col": "value_over_replacement",
        "label_map": {"0": "below_average", "1": "above_average"},
        "description": (
            "NBA draft picks from 1989 to 2021. "
            "The task is to predict whether a draft pick provided above-average career value."
        ),
        "feature_descriptions": {
            "year":                   "The year of the NBA draft.",
            "overall_pick":           "The overall pick number in the draft.",
            "team":                   "The NBA franchise that selected the player.",
            "years_active":           "The number of seasons the player was active in the NBA.",
            "value_over_replacement": "Whether the player's career value was above average.",
        },
    },
}

ds_config  = CARTE_DATASETS[DATASET]
TARGET_COL = ds_config["target_col"]
LABEL_MAP  = ds_config.get("label_map")
DESCRIPTION = ds_config["description"]
FEAT_DESCS  = ds_config["feature_descriptions"]
print(f"Dataset : {DATASET}")
print(f"Target  : {TARGET_COL}")

## 4. Load dataset from HuggingFace (CARTE benchmark)

In [ ]:
from huggingface_hub import hf_hub_download

print(f"Downloading {DATASET} from CARTE benchmark...")
path = hf_hub_download(
    "inria-soda/carte-benchmark",
    f"data_carte/{DATASET}/raw.parquet",
    repo_type="dataset",
)
df = pd.read_parquet(path).dropna(how="all")
df = df.dropna(subset=[TARGET_COL])

if df[TARGET_COL].dtype != object:
    df[TARGET_COL] = df[TARGET_COL].apply(
        lambda v: str(int(v)) if pd.notna(v) and float(v) == int(float(v)) else str(v))
else:
    df[TARGET_COL] = df[TARGET_COL].astype(str)

if LABEL_MAP:
    df[TARGET_COL] = df[TARGET_COL].map(LABEL_MAP).fillna(df[TARGET_COL])

if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)

print(f"Rows: {len(df)}  Columns: {len(df.columns)}")
print(f"\nClass distribution:")
print(df[TARGET_COL].value_counts())
df.head(3)

## 5. Train / test split

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=SEED, stratify=y)
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=SEED)

X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 6. Build feature specs

Feature specs are inferred automatically from the DataFrame (dtype + cardinality).  
Descriptions are pulled from the dataset registry above.  
**The target column is placed last.**

In [ ]:
from aspire.model import Feature

MAX_CAT_UNIQUE = 1000

def build_feature_specs(df, target_col, feat_descs, max_cat=MAX_CAT_UNIQUE):
    non_target, target_feat = [], None
    for col in df.columns:
        col_data = df[col].dropna()
        if len(col_data) < 5:
            continue
        desc = feat_descs.get(col, f"{col}: a feature in this dataset.")
        feat = None
        if col_data.dtype in ["object", "category", "bool"]:
            uniq = sorted(col_data.astype(str).unique())
            if 2 <= len(uniq) <= max_cat:
                feat = {"name": col, "description": desc, "dtype": "categorical", "choices": uniq}
        else:
            numeric = pd.to_numeric(col_data, errors="coerce").dropna()
            if len(numeric) < len(col_data) * 0.5:
                continue
            uniq_vals = numeric.unique()
            if len(uniq_vals) <= 15 and all(v == int(v) for v in uniq_vals if np.isfinite(v)):
                choices = sorted([str(int(v)) for v in uniq_vals if np.isfinite(v)])
                feat = {"name": col, "description": desc, "dtype": "categorical", "choices": choices} \
                    if 2 <= len(choices) <= max_cat \
                    else {"name": col, "description": desc, "dtype": "continuous",
                          "value_range": (float(numeric.min()), float(numeric.max()))}
            else:
                feat = {"name": col, "description": desc, "dtype": "continuous",
                        "value_range": (float(numeric.min()), float(numeric.max()))}
        if feat is None:
            continue
        if col == target_col:
            target_feat = feat
        else:
            non_target.append(feat)
    if target_feat is not None:
        non_target.append(target_feat)
    return non_target

full_df = pd.concat([X_train, y_train.rename(TARGET_COL)], axis=1)
feature_specs = build_feature_specs(full_df, TARGET_COL, FEAT_DESCS)

assert feature_specs[-1]["name"] == TARGET_COL, "Target must be last!"
print(f"Feature specs: {len(feature_specs)} features (target: '{TARGET_COL}')")
print(f"Classes: {feature_specs[-1]['choices']}")

In [ ]:
print("=" * 60)
print("  ASPIRE v2 Finetuning")
print("=" * 60)

t0 = time.time()
model = AspireModel.from_pretrained(
    CKPT,
    filename=CKPT_FILE,
    device=DEVICE,
    feature_specs=feature_specs,
    dataset_context=DESCRIPTION,
    model_type="aspire_lite",
)
print(f"Checkpoint loaded from: {CKPT}")

model.fit(
    X_train, y_train,
    finetune_mode="v2",
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    num_support=NUM_SUPPORT,
    test_fraction=0.15,
    random_state=SEED,
    show_progress=True,
)
fit_time = time.time() - t0
print(f"Finetune time: {fit_time:.1f}s")

## 8. Evaluate ASPIRE on the test set

In [ ]:
print("Scoring on test set...")
aspire_preds = model.predict(X_test)
aspire_acc   = accuracy_score(y_test, aspire_preds)
aspire_f1m   = f1_score(y_test, aspire_preds, average="macro",    zero_division=0)
aspire_f1w   = f1_score(y_test, aspire_preds, average="weighted", zero_division=0)

print(f"  Accuracy        : {aspire_acc:.4f}")
print(f"  F1 macro        : {aspire_f1m:.4f}")
print(f"  F1 weighted     : {aspire_f1w:.4f}")
print()
print(classification_report(y_test, aspire_preds, zero_division=0))

## 9. XGBoost baseline

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

print("=" * 60)
print("  XGBoost Baseline")
print("=" * 60)

X_tr = pd.get_dummies(X_train).astype(float)
X_te = pd.get_dummies(X_test).astype(float)
X_tr, X_te = X_tr.align(X_te, join="left", axis=1, fill_value=0)

le = LabelEncoder()
y_tr = le.fit_transform(y_train.astype(str))
y_te = le.transform(y_test.astype(str))
n_cls = len(le.classes_)

t0 = time.time()
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="multi:softmax" if n_cls > 2 else "binary:logistic",
    eval_metric="mlogloss", random_state=SEED,
    **(dict(num_class=n_cls) if n_cls > 2 else {}),
)
xgb.fit(X_tr, y_tr, verbose=False)
xgb_time = time.time() - t0

xgb_preds_enc = xgb.predict(X_te)
xgb_preds = le.inverse_transform(xgb_preds_enc)
xgb_acc   = accuracy_score(y_test, xgb_preds)
xgb_f1m   = f1_score(y_test, xgb_preds, average="macro",    zero_division=0)
xgb_f1w   = f1_score(y_test, xgb_preds, average="weighted", zero_division=0)

print(f"  Accuracy        : {xgb_acc:.4f}")
print(f"  F1 macro        : {xgb_f1m:.4f}")
print(f"  F1 weighted     : {xgb_f1w:.4f}")
print(f"  Time            : {xgb_time:.1f}s")

## 10. Summary

In [ ]:
print("=" * 60)
print("  Summary")
print("=" * 60)
print(f"  {'Model':<20} {'Accuracy':>10} {'F1 macro':>10} {'F1 weighted':>12}")
print(f"  {'─'*20} {'─'*10} {'─'*10} {'─'*12}")
print(f"  {'ASPIRE v2':<20} {aspire_acc:>10.4f} {aspire_f1m:>10.4f} {aspire_f1w:>12.4f}")
print(f"  {'XGBoost':<20} {xgb_acc:>10.4f} {xgb_f1m:>10.4f} {xgb_f1w:>12.4f}")

winner = "ASPIRE v2" if aspire_f1m > xgb_f1m else "XGBoost"
delta  = abs(aspire_f1m - xgb_f1m)
print(f"\n  {winner} wins by {delta:.4f} F1 macro.")

results = {
    "aspire_v2": {"accuracy": aspire_acc, "f1_macro": aspire_f1m, "f1_weighted": aspire_f1w, "fit_time_s": fit_time},
    "xgboost":           {"accuracy": xgb_acc,   "f1_macro": xgb_f1m,   "f1_weighted": xgb_f1w,   "fit_time_s": xgb_time},
}
results